In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.sales_monthly AS
SELECT o.purchase_month,
       p.category_en AS category,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(i.price) AS revenue,
       SUM(i.freight_value) AS freight,
       ROUND(SUM(i.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM workspace.silver.orders o
JOIN workspace.silver.order_items i USING (order_id)
JOIN workspace.silver.products p USING (product_id)
WHERE o.order_status = 'delivered'
GROUP BY ALL;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.delivery_by_state AS
WITH r AS (SELECT order_id, AVG(review_score) AS score FROM workspace.silver.reviews GROUP BY order_id)
SELECT c.customer_state AS state,
       COUNT(*) AS delivered_orders,
       ROUND(AVG(o.delivery_days), 1) AS avg_delivery_days,
       ROUND(AVG(CASE WHEN o.is_late THEN 1 ELSE 0 END) * 100, 1) AS late_pct,
       ROUND(AVG(r.score), 2) AS avg_review_score
FROM workspace.silver.orders o
JOIN workspace.silver.customers c USING (customer_id)
LEFT JOIN r USING (order_id)
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
GROUP BY ALL;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.seller_performance AS
WITH r AS (SELECT order_id, AVG(review_score) AS score FROM workspace.silver.reviews GROUP BY order_id)
SELECT i.seller_id, s.seller_state,
       COUNT(DISTINCT i.order_id) AS orders,
       SUM(i.price) AS revenue,
       ROUND(AVG(r.score), 2) AS avg_review_score,
       ROUND(AVG(CASE WHEN o.is_late THEN 1 ELSE 0 END) * 100, 1) AS late_pct
FROM workspace.silver.order_items i
JOIN workspace.silver.orders o USING (order_id)
JOIN workspace.silver.sellers s USING (seller_id)
LEFT JOIN r USING (order_id)
WHERE o.order_status = 'delivered'
GROUP BY ALL
HAVING orders >= 10;

In [0]:
%sql
SELECT CASE WHEN o.is_late THEN 'Late' ELSE 'On time' END AS delivery,
       COUNT(*) AS orders,
       ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM workspace.silver.orders o
JOIN workspace.silver.reviews r USING (order_id)
WHERE o.order_status = 'delivered' AND o.is_late IS NOT NULL
GROUP BY ALL;